# Black-Litterman Research Lab v16 — Improved View Construction (Fixed)

This notebook fixes the missing `get_rebalance_dates` helper and keeps the improved BL view engine.


## How to read this notebook

1. Summary table
2. Winner board
3. Portfolio value vs contributions
4. Drawdown
5. Benchmark comparison
6. Latest weights + sleeve summary
7. Concentration and turnover
8. View diagnostics


In [ ]:
!pip -q install yfinance PyPortfolioOpt plotly pandas numpy scipy scikit-learn openai


In [ ]:
import warnings, os, json
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import yfinance as yf
import plotly.express as px
import plotly.graph_objects as go

from pypfopt.risk_models import CovarianceShrinkage
from pypfopt.hierarchical_portfolio import HRPOpt
from pypfopt.black_litterman import BlackLittermanModel
from pypfopt.efficient_frontier import EfficientFrontier

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 200)


## 1) Configuration


In [ ]:
BIST_TICKERS = ['AEFES.IS', 'AGHOL.IS', 'AKBNK.IS', 'AKFYE.IS', 'AKSA.IS', 'AKSEN.IS', 'ALARK.IS', 'ARCLK.IS', 'ASELS.IS', 'ASTOR.IS', 'BIMAS.IS', 'BOBET.IS', 'CCOLA.IS', 'CIMSA.IS', 'CWENE.IS', 'DOAS.IS', 'DOHOL.IS', 'ECILC.IS', 'EGEEN.IS', 'EKGYO.IS', 'ENERY.IS', 'ENJSA.IS', 'ENKAI.IS', 'EREGL.IS', 'EUPWR.IS', 'FROTO.IS', 'GARAN.IS', 'GESAN.IS', 'GUBRF.IS', 'HALKB.IS', 'HEKTS.IS', 'ISCTR.IS', 'ISMEN.IS', 'KAYSE.IS', 'KCAER.IS', 'KCHOL.IS', 'KLSER.IS', 'KOZAA.IS', 'KOZAL.IS', 'KRDMD.IS', 'MAVI.IS', 'MGROS.IS', 'MIATK.IS', 'ODAS.IS', 'OTKAR.IS', 'OYAKC.IS', 'PETKM.IS', 'PGSUS.IS', 'REEDR.IS', 'SASA.IS', 'SAHOL.IS', 'SDTTR.IS', 'SISE.IS', 'SKBNK.IS', 'SMRTG.IS', 'SOKM.IS', 'TAVHL.IS', 'TCELL.IS', 'THYAO.IS', 'TKFEN.IS', 'TOASO.IS', 'TSKB.IS', 'TTKOM.IS', 'TTRAK.IS', 'TUPRS.IS', 'ULKER.IS', 'VAKBN.IS', 'VESBE.IS', 'VESTL.IS', 'YKBNK.IS', 'ZOREN.IS']
FX_TICKERS = {"USDTRY": "USDTRY=X", "EURTRY": "EURTRY=X"}
METAL_USD_TICKERS = {"ALTIN_USD": "GC=F", "GUMUS_USD": "SI=F", "PLATIN_USD": "PL=F"}

BACKTEST = {
    "download_period": "10y",
    "min_history_days": 180,
    "lookback_days": 252,
    "test_months": 6,
    "monthly_contribution_try": 25000,
}

STRATEGY_CFG = {
    "strategies": ["Equal", "HRP", "BL", "HYBRID"],
    "max_weight": 0.10,
    "hybrid_alpha": 0.20,
    "gold_label": "ALTIN_TRY",
    "gold_min": 0.05,
    "gold_max": 0.20,
}

VIEW_CFG = {
    "mom_3m_days": 63,
    "mom_6m_days": 126,
    "vol_days": 63,
    "shrink_factor": 0.025,
    "view_cap": 0.04,
    "top_k_positive": 5,
    "top_k_negative": 5,
    "min_confidence": 0.35,
    "max_confidence": 0.95,
}


## 2) Helpers


In [ ]:
def try_fmt(x):
    if pd.isna(x):
        return "-"
    return f"₺{x:,.0f}".replace(",", "X").replace(".", ",").replace("X", ".")

def pct_fmt(x):
    if pd.isna(x):
        return "-"
    return f"{x*100:.2f}%"

def safe_asset_name(symbol):
    return symbol.replace(".IS", "").replace("=X", "").replace("=F", "")

def normalize_prices(df):
    return df / df.iloc[0] * 100

def fetch_close_series(symbol, period):
    df = yf.download(symbol, period=period, auto_adjust=True, progress=False)
    if df is None or len(df) == 0:
        return None
    if isinstance(df.columns, pd.MultiIndex):
        level0 = list(df.columns.get_level_values(0))
        close = df.xs("Close", axis=1, level=0) if "Close" in level0 else df.iloc[:, :1]
    else:
        close = df[["Close"]] if "Close" in df.columns else df.iloc[:, :1]
    if isinstance(close, pd.DataFrame):
        if close.shape[1] == 0:
            return None
        close = close.iloc[:, 0]
    s = pd.Series(close).dropna()
    s.name = symbol
    return s if len(s) else None

def clean_returns(df):
    rets = df.pct_change().replace([np.inf, -np.inf], np.nan).dropna(how="any")
    if rets.empty:
        return rets
    valid_cols = [c for c in rets.columns if np.isfinite(rets[c]).all() and rets[c].std() > 1e-10]
    if len(valid_cols) == 0:
        return pd.DataFrame(index=rets.index)
    return rets[valid_cols]

def normalize_weights(w, index):
    w = pd.Series(w).reindex(index).fillna(0.0).astype(float)
    s = w.sum()
    if s <= 0 or not np.isfinite(s):
        return pd.Series(1 / len(index), index=index)
    return w / s

def band_gold(weights, gold_label, gold_min, gold_max):
    w = weights.copy().astype(float)
    if gold_label not in w.index:
        return normalize_weights(w, w.index)
    g = float(w[gold_label])
    tg = min(max(g, gold_min), gold_max)
    if abs(tg - g) < 1e-12:
        return normalize_weights(w, w.index)
    rest = w.drop(gold_label)
    if rest.sum() <= 0:
        w[:] = 0.0
        w[gold_label] = 1.0
        return w
    rest = rest / rest.sum()
    w[gold_label] = tg
    w.loc[rest.index] = (1 - tg) * rest
    return normalize_weights(w, w.index)

def sleeve_of(asset):
    if asset in ["USDTRY", "EURTRY"]:
        return "FX"
    if asset in ["ALTIN_TRY", "GUMUS_TRY", "PLATIN_TRY"]:
        return "METALS"
    return "BIST"

def zscore_series(s):
    s = pd.Series(s, dtype=float)
    std = s.std(ddof=0)
    if len(s) < 2 or pd.isna(std) or std < 1e-12:
        return pd.Series(0.0, index=s.index)
    return (s - s.mean()) / std

def get_rebalance_dates(prices, months):
    monthly = prices.resample("MS").first().dropna()
    month_starts = monthly.index[-months:]
    dates = []
    for dt in month_starts:
        idx = prices.index.searchsorted(dt)
        if idx < len(prices.index):
            dates.append(prices.index[idx])
    return pd.Index(pd.unique(pd.Index(dates)))


## 3) Data loading and coverage


In [ ]:
rows = []
series_list = []
for sym in BIST_TICKERS:
    s = fetch_close_series(sym, BACKTEST["download_period"])
    rows.append({"source": sym, "asset": safe_asset_name(sym), "group": "BIST", "n": 0 if s is None else len(s), "start": None if s is None else str(s.index.min().date()), "end": None if s is None else str(s.index.max().date()), "status": "ok" if s is not None else "empty"})
    if s is not None and len(s) >= BACKTEST["min_history_days"]:
        s = s.copy(); s.name = safe_asset_name(sym); series_list.append(s)
fx_series = {}
for label, sym in FX_TICKERS.items():
    s = fetch_close_series(sym, BACKTEST["download_period"])
    fx_series[label] = s
    rows.append({"source": sym, "asset": label, "group": "FX", "n": 0 if s is None else len(s), "start": None if s is None else str(s.index.min().date()), "end": None if s is None else str(s.index.max().date()), "status": "ok" if s is not None else "empty"})
    if s is not None and len(s) >= BACKTEST["min_history_days"]:
        s = s.copy(); s.name = label; series_list.append(s)
metal_usd = {}
for label, sym in METAL_USD_TICKERS.items():
    s = fetch_close_series(sym, BACKTEST["download_period"])
    metal_usd[label] = s
    rows.append({"source": sym, "asset": label, "group": "METAL_USD", "n": 0 if s is None else len(s), "start": None if s is None else str(s.index.min().date()), "end": None if s is None else str(s.index.max().date()), "status": "ok" if s is not None else "empty"})
usdtry = fx_series.get("USDTRY")
for try_label, usd_label in [("ALTIN_TRY", "ALTIN_USD"), ("GUMUS_TRY", "GUMUS_USD"), ("PLATIN_TRY", "PLATIN_USD")]:
    base = metal_usd.get(usd_label)
    if base is not None and usdtry is not None:
        s = (base * usdtry).dropna()
        s.name = try_label
        rows.append({"source": f"{usd_label} * USDTRY", "asset": try_label, "group": "METAL_TRY", "n": len(s), "start": str(s.index.min().date()), "end": str(s.index.max().date()), "status": "ok"})
        if len(s) >= BACKTEST["min_history_days"]:
            series_list.append(s)
coverage = pd.DataFrame(rows)
prices = pd.concat(series_list, axis=1).sort_index().ffill()
common_start = prices.apply(lambda s: s.first_valid_index()).max()
prices = prices.loc[common_start:].dropna()
returns = prices.pct_change().dropna()
lookback = min(BACKTEST["lookback_days"], max(60, len(prices) - 40))
months = BACKTEST["test_months"]
display(coverage.head())
print("Common start:", common_start)
print("Usable assets:", prices.shape[1])
print("Total days:", len(prices), "lookback:", lookback, "test_months:", months)


In [ ]:
coverage_counts = coverage.groupby(["group", "status"]).size().reset_index(name="count")
fig = px.bar(coverage_counts, x="group", y="count", color="status", barmode="group", title="Data coverage by group")
fig.show()


## 4) Market context


In [ ]:
bist_cols = [c for c in prices.columns if c not in ["USDTRY", "EURTRY", "ALTIN_TRY", "GUMUS_TRY", "PLATIN_TRY"]]
bist_basket = (1 + returns[bist_cols].mean(axis=1)).cumprod() if len(bist_cols) else pd.Series(dtype=float)
if len(bist_basket):
    bist_basket.name = "BIST_BASKET"
bench = pd.concat([bist_basket, prices[["USDTRY", "EURTRY", "ALTIN_TRY", "GUMUS_TRY", "PLATIN_TRY"]]], axis=1).dropna()
bench_norm = normalize_prices(bench.tail(504))
px.line(bench_norm, x=bench_norm.index, y=bench_norm.columns, title="Last ~2 years benchmark view (normalized)").show()
px.imshow(bench.pct_change().dropna().corr(), text_auto=True, aspect="auto", zmin=-1, zmax=1, color_continuous_scale="RdBu_r", title="Benchmark correlation heatmap").show()


## 5) Improved BL view construction


In [ ]:
def build_view_table(price_window):
    px_window = price_window.copy()
    rows = []
    for asset in px_window.columns:
        s = px_window[asset].dropna()
        min_len = max(VIEW_CFG["mom_6m_days"] + 1, VIEW_CFG["vol_days"] + 1)
        if len(s) < min_len:
            rows.append({"asset": asset, "sleeve": sleeve_of(asset), "mom_3m": np.nan, "mom_6m": np.nan, "vol_3m": np.nan})
            continue
        mom_3m = s.iloc[-1] / s.iloc[-VIEW_CFG["mom_3m_days"]] - 1 if len(s) > VIEW_CFG["mom_3m_days"] else np.nan
        mom_6m = s.iloc[-1] / s.iloc[-VIEW_CFG["mom_6m_days"]] - 1 if len(s) > VIEW_CFG["mom_6m_days"] else np.nan
        vol_3m = s.pct_change().tail(VIEW_CFG["vol_days"]).std() * np.sqrt(252)
        rows.append({"asset": asset, "sleeve": sleeve_of(asset), "mom_3m": mom_3m, "mom_6m": mom_6m, "vol_3m": vol_3m})
    vt = pd.DataFrame(rows).set_index("asset")
    vt["z_mom_3m"] = 0.0
    vt["z_mom_6m"] = 0.0
    vt["z_vol_3m"] = 0.0
    for sleeve, idx in vt.groupby("sleeve").groups.items():
        vt.loc[idx, "z_mom_3m"] = zscore_series(vt.loc[idx, "mom_3m"].fillna(0))
        vt.loc[idx, "z_mom_6m"] = zscore_series(vt.loc[idx, "mom_6m"].fillna(0))
        vt.loc[idx, "z_vol_3m"] = zscore_series(vt.loc[idx, "vol_3m"].fillna(vt.loc[idx, "vol_3m"].median()))
    vt["raw_score"] = 0.45 * vt["z_mom_3m"].fillna(0) + 0.45 * vt["z_mom_6m"].fillna(0) - 0.10 * vt["z_vol_3m"].fillna(0)
    vt["shrunk_score"] = VIEW_CFG["shrink_factor"] * vt["raw_score"]
    vt["final_view"] = 0.0
    pos_assets = vt[vt["shrunk_score"] > 0].sort_values("shrunk_score", ascending=False).head(VIEW_CFG["top_k_positive"]).index.tolist()
    neg_assets = vt[vt["shrunk_score"] < 0].sort_values("shrunk_score", ascending=True).head(VIEW_CFG["top_k_negative"]).index.tolist()
    keep = set(pos_assets + neg_assets)
    if keep:
        vt.loc[list(keep), "final_view"] = vt.loc[list(keep), "shrunk_score"].clip(-VIEW_CFG["view_cap"], VIEW_CFG["view_cap"])
    max_abs = max(vt["raw_score"].abs().max(), 1e-9)
    scaled_abs = (vt["raw_score"].abs() / max_abs).clip(0, 1)
    vt["confidence"] = VIEW_CFG["min_confidence"] + (VIEW_CFG["max_confidence"] - VIEW_CFG["min_confidence"]) * scaled_abs
    vt.loc[vt["final_view"] == 0, "confidence"] = VIEW_CFG["min_confidence"]
    return vt.reset_index()

def views_from_price_window(price_window):
    vt = build_view_table(price_window)
    return dict(zip(vt["asset"], vt["final_view"])), vt

def build_omega_from_confidence(S, vt):
    vt = vt.set_index("asset").reindex(S.index)
    base_var = np.diag(S.values)
    conf = vt["confidence"].fillna(VIEW_CFG["min_confidence"]).clip(VIEW_CFG["min_confidence"], VIEW_CFG["max_confidence"]).values
    omega_diag = base_var * (1.0 / conf)
    return np.diag(omega_diag)

latest_view_table = build_view_table(prices.tail(lookback))
latest_view_table["view_pct"] = latest_view_table["final_view"].map(pct_fmt)
latest_view_table["confidence_pct"] = latest_view_table["confidence"].map(lambda x: f"{x*100:.0f}%")
display(latest_view_table.sort_values("final_view", ascending=False).head(20))
display(latest_view_table.sort_values("final_view", ascending=True).head(20))


## 6) Strategy functions


In [ ]:
def equal_weight(columns):
    return pd.Series(1 / len(columns), index=columns)

def inverse_vol_weight(price_window):
    rets = clean_returns(price_window)
    if rets.empty or rets.shape[1] == 0:
        return equal_weight(price_window.columns)
    iv = 1 / rets.std().replace(0, np.nan)
    iv = iv.replace([np.inf, -np.inf], np.nan).dropna()
    if iv.empty:
        return equal_weight(price_window.columns)
    w = pd.Series(0.0, index=price_window.columns)
    w.loc[iv.index] = iv / iv.sum()
    return normalize_weights(w, price_window.columns)

def hrp_weights_safe(price_window):
    rets = clean_returns(price_window)
    if rets.shape[0] < 2 or rets.shape[1] < 2:
        return inverse_vol_weight(price_window)
    try:
        hrp = HRPOpt(rets)
        w_sub = pd.Series(hrp.optimize()).reindex(rets.columns).fillna(0.0)
        w = pd.Series(0.0, index=price_window.columns)
        w.loc[w_sub.index] = w_sub
        return normalize_weights(w, price_window.columns)
    except Exception as e:
        print("HRP failed, inverse-vol used:", str(e))
        return inverse_vol_weight(price_window)

def bl_weights_safe(price_window):
    rets = clean_returns(price_window)
    if rets.shape[0] < 2 or rets.shape[1] < 2:
        w = inverse_vol_weight(price_window)
        diag = pd.DataFrame({"asset": list(price_window.columns), "final_view": [0.0]*len(price_window.columns), "confidence": VIEW_CFG["min_confidence"], "view_source": "insufficient_data"})
        return w, diag
    try:
        S = CovarianceShrinkage(rets, returns_data=True).ledoit_wolf()
        views, vt = views_from_price_window(price_window)
        omega = build_omega_from_confidence(S, vt)
        bl = BlackLittermanModel(S, pi="equal", absolute_views=views, omega=omega)
        post = bl.bl_returns()
        ef = EfficientFrontier(post, S, weight_bounds=(0, STRATEGY_CFG["max_weight"]))
        ef.max_sharpe(risk_free_rate=0.0)
        w = pd.Series(ef.clean_weights()).reindex(price_window.columns).fillna(0.0)
        w = normalize_weights(w, price_window.columns)
        w = band_gold(w, STRATEGY_CFG["gold_label"], STRATEGY_CFG["gold_min"], STRATEGY_CFG["gold_max"])
        diag = vt.copy(); diag["view_source"] = "improved_rules"
        return w, diag
    except Exception as e:
        print("BL failed, inverse-vol used:", str(e))
        w = inverse_vol_weight(price_window)
        _, vt = views_from_price_window(price_window)
        vt["view_source"] = "improved_rules_fallback"
        return w, vt

def hybrid_weights(price_window):
    w_hrp = hrp_weights_safe(price_window)
    w_bl, diag = bl_weights_safe(price_window)
    a = STRATEGY_CFG["hybrid_alpha"]
    w = (1 - a) * w_hrp + a * w_bl
    w = normalize_weights(w, price_window.columns)
    w = w.clip(lower=0, upper=STRATEGY_CFG["max_weight"])
    w = normalize_weights(w, price_window.columns)
    w = band_gold(w, STRATEGY_CFG["gold_label"], STRATEGY_CFG["gold_min"], STRATEGY_CFG["gold_max"])
    return w, diag

def compute_weights(strategy_name, price_window):
    if strategy_name == "Equal":
        w = equal_weight(price_window.columns)
        w = w.clip(upper=STRATEGY_CFG["max_weight"])
        w = normalize_weights(w, price_window.columns)
        return w, None
    if strategy_name == "HRP":
        w = hrp_weights_safe(price_window)
        w = w.clip(upper=STRATEGY_CFG["max_weight"])
        w = normalize_weights(w, price_window.columns)
        return w, None
    if strategy_name == "BL":
        return bl_weights_safe(price_window)
    if strategy_name == "HYBRID":
        return hybrid_weights(price_window)
    raise ValueError(f"Unknown strategy: {strategy_name}")


## 7) Monthly walk-forward backtest


In [ ]:
def run_strategy(strategy_name, prices, lookback, months):
    rebalance_dates = get_rebalance_dates(prices, months)
    cash = 0.0
    shares = pd.Series(0.0, index=prices.columns)
    hist, rebs, diags = [], [], []
    for i, dt in enumerate(rebalance_dates):
        cash += BACKTEST["monthly_contribution_try"]
        window = prices.loc[:dt].tail(lookback)
        weights, diag = compute_weights(strategy_name, window)
        current = prices.loc[dt]
        total = float((shares * current).sum()) + cash
        shares = (weights * total) / current
        cash = 0.0
        nxt = rebalance_dates[i + 1] if i + 1 < len(rebalance_dates) else prices.index[-1]
        path = prices.loc[(prices.index >= dt) & (prices.index <= nxt)]
        vals = path.mul(shares, axis=1).sum(axis=1)
        for t, v in vals.items():
            hist.append({"date": t, "strategy": strategy_name, "portfolio_value": float(v)})
        row = {"date": dt, "strategy": strategy_name, "capital_after_contribution": total}
        for c, x in weights.items():
            row[f"weight_{c}"] = float(x)
        rebs.append(row)
        if diag is not None:
            tmp = diag.copy(); tmp["date"] = dt; tmp["strategy"] = strategy_name; diags.append(tmp)
    hist_df = pd.DataFrame(hist).drop_duplicates(["date", "strategy"])
    weights_df = pd.DataFrame(rebs)
    diag_df = pd.concat(diags, ignore_index=True) if diags else pd.DataFrame()
    return hist_df, weights_df, diag_df

hist_parts, weight_parts, diag_parts = [], [], []
for s in STRATEGY_CFG["strategies"]:
    h, w, d = run_strategy(s, prices, lookback, months)
    hist_parts.append(h); weight_parts.append(w)
    if not d.empty:
        diag_parts.append(d)
hist_df = pd.concat(hist_parts, ignore_index=True)
weights_df = pd.concat(weight_parts, ignore_index=True)
diag_df = pd.concat(diag_parts, ignore_index=True) if diag_parts else pd.DataFrame()
display(hist_df.tail())
display(weights_df.tail())


## 8) Summary table


In [ ]:
summary_rows = []
contributed = BACKTEST["monthly_contribution_try"] * months
for s in STRATEGY_CFG["strategies"]:
    sub = hist_df[hist_df["strategy"] == s].sort_values("date").copy()
    sub["ret"] = sub["portfolio_value"].pct_change().fillna(0.0)
    dd = sub["portfolio_value"] / sub["portfolio_value"].cummax() - 1
    monthly_ret = sub.set_index("date")["portfolio_value"].resample("M").last().pct_change().dropna()
    summary_rows.append({"Strategy": s, "Contributed": try_fmt(contributed), "Ending Value": try_fmt(sub["portfolio_value"].iloc[-1]), "Net Profit": try_fmt(sub["portfolio_value"].iloc[-1] - contributed), "TWR_like": pct_fmt(sub["portfolio_value"].iloc[-1] / contributed - 1), "Ann Vol": pct_fmt(sub["ret"].std() * np.sqrt(252)), "Sharpe": f"{(sub['ret'].mean() / (sub['ret'].std() + 1e-9) * np.sqrt(252)):.2f}", "MaxDD": pct_fmt(dd.min()), "Avg Monthly": pct_fmt(monthly_ret.mean() if len(monthly_ret) else np.nan)})
summary = pd.DataFrame(summary_rows)
display(summary)


## 9) Winner board


In [ ]:
summary_numeric = []
for s in STRATEGY_CFG["strategies"]:
    sub = hist_df[hist_df["strategy"] == s].sort_values("date").copy()
    sub["ret"] = sub["portfolio_value"].pct_change().fillna(0.0)
    dd = sub["portfolio_value"] / sub["portfolio_value"].cummax() - 1
    summary_numeric.append({"strategy": s, "ending_value": sub["portfolio_value"].iloc[-1], "maxdd": dd.min(), "sharpe": (sub["ret"].mean() / (sub["ret"].std() + 1e-9) * np.sqrt(252))})
summary_numeric = pd.DataFrame(summary_numeric)
winner_board = pd.DataFrame([{"category": "Highest Final Value", "winner": summary_numeric.loc[summary_numeric["ending_value"].idxmax(), "strategy"], "value": try_fmt(summary_numeric["ending_value"].max())}, {"category": "Lowest Max Drawdown", "winner": summary_numeric.loc[summary_numeric["maxdd"].idxmax(), "strategy"], "value": pct_fmt(summary_numeric["maxdd"].max())}, {"category": "Best Sharpe", "winner": summary_numeric.loc[summary_numeric["sharpe"].idxmax(), "strategy"], "value": f"{summary_numeric['sharpe'].max():.2f}"}])
display(winner_board)


## 10) Portfolio value vs contributions


In [ ]:
rebal_dates = get_rebalance_dates(prices, months)
contrib_series = pd.Series(np.arange(1, len(rebal_dates) + 1) * BACKTEST["monthly_contribution_try"], index=rebal_dates).reindex(hist_df["date"].sort_values().unique(), method="ffill")
fig = go.Figure()
for s in STRATEGY_CFG["strategies"]:
    sub = hist_df[hist_df["strategy"] == s].sort_values("date")
    fig.add_trace(go.Scatter(x=sub["date"], y=sub["portfolio_value"], mode="lines", name=s))
fig.add_trace(go.Scatter(x=contrib_series.index, y=contrib_series.values, mode="lines", name="Contributed Cash", line=dict(dash="dash")))
fig.update_layout(title="Portfolio value vs cumulative contributed cash", template="plotly_white")
fig.update_yaxes(tickprefix="₺", separatethousands=True)
fig.show()


## 11) Performance comparison


In [ ]:
px.line(hist_df, x="date", y="portfolio_value", color="strategy", title="Portfolio value by strategy").show()
dd_parts = []
for s in STRATEGY_CFG["strategies"]:
    sub = hist_df[hist_df["strategy"] == s].sort_values("date").copy()
    sub["drawdown"] = sub["portfolio_value"] / sub["portfolio_value"].cummax() - 1
    dd_parts.append(sub[["date", "strategy", "drawdown"]])
dd_df = pd.concat(dd_parts, ignore_index=True)
px.line(dd_df, x="date", y="drawdown", color="strategy", title="Drawdown by strategy").show()
mr_parts = []
for s in STRATEGY_CFG["strategies"]:
    sub = hist_df[hist_df["strategy"] == s].sort_values("date").copy().set_index("date")
    monthly = sub["portfolio_value"].resample("M").last().pct_change().dropna()
    mr_parts.append(pd.DataFrame({"date": monthly.index, "strategy": s, "monthly_return": monthly.values}))
mr_df = pd.concat(mr_parts, ignore_index=True)
px.bar(mr_df, x="date", y="monthly_return", color="strategy", barmode="group", title="Monthly portfolio returns").show()


## 12) Benchmark comparison


In [ ]:
def run_fixed_mix(name, prices, alloc_map, months):
    rebalance_dates = get_rebalance_dates(prices, months)
    cash = 0.0
    shares = pd.Series(0.0, index=prices.columns)
    hist = []
    for i, dt in enumerate(rebalance_dates):
        cash += BACKTEST["monthly_contribution_try"]
        w = pd.Series(0.0, index=prices.columns)
        for asset, wt in alloc_map.items():
            if asset == "BIST_BASKET":
                basket_assets = [c for c in prices.columns if c in bist_cols]
                if len(basket_assets):
                    w.loc[basket_assets] += wt / len(basket_assets)
            elif asset in prices.columns:
                w.loc[asset] += wt
        w = normalize_weights(w, prices.columns)
        current = prices.loc[dt]
        total = float((shares * current).sum()) + cash
        shares = (w * total) / current
        cash = 0.0
        nxt = rebalance_dates[i + 1] if i + 1 < len(rebalance_dates) else prices.index[-1]
        path = prices.loc[(prices.index >= dt) & (prices.index <= nxt)]
        vals = path.mul(shares, axis=1).sum(axis=1)
        for t, v in vals.items():
            hist.append({"date": t, "strategy": name, "portfolio_value": float(v)})
    return pd.DataFrame(hist).drop_duplicates(["date", "strategy"])
benchmark_defs = {"BIST_BASKET": {"BIST_BASKET": 1.0}, "USD_ONLY": {"USDTRY": 1.0}, "GOLD_ONLY": {"ALTIN_TRY": 1.0}, "USD_GOLD_50_50": {"USDTRY": 0.5, "ALTIN_TRY": 0.5}, "SIMPLE_60_20_20": {"BIST_BASKET": 0.6, "USDTRY": 0.2, "ALTIN_TRY": 0.2}}
bench_hist_parts = []
for n, alloc in benchmark_defs.items():
    bench_hist_parts.append(run_fixed_mix(n, prices, alloc, months))
bench_hist_df = pd.concat(bench_hist_parts, ignore_index=True)
compare_df = pd.concat([hist_df, bench_hist_df], ignore_index=True)
px.line(compare_df, x="date", y="portfolio_value", color="strategy", title="Strategies vs simple benchmarks").show()


## 13) Latest weights and sleeve summary


In [ ]:
last_weights = weights_df.sort_values("date").groupby("strategy").tail(1).copy()
latest_long = last_weights.melt(id_vars=["date", "strategy", "capital_after_contribution"], var_name="asset", value_name="weight")
latest_long = latest_long[latest_long["asset"].str.startswith("weight_")].copy()
latest_long["asset"] = latest_long["asset"].str.replace("weight_", "", regex=False)
latest_long = latest_long[latest_long["weight"] > 0].copy()
fig = px.bar(latest_long.sort_values(["strategy", "weight"], ascending=[True, False]), x="asset", y="weight", color="strategy", facet_row="strategy", title="Latest rebalance weights by strategy")
fig.for_each_annotation(lambda a: a.update(text=a.text.split('=')[-1]))
fig.show()
latest_long["sleeve"] = latest_long["asset"].map(sleeve_of)
sleeve_summary = latest_long.groupby(["strategy", "sleeve"])["weight"].sum().reset_index()
sleeve_pivot = sleeve_summary.pivot(index="strategy", columns="sleeve", values="weight").fillna(0.0)
for c in sleeve_pivot.columns:
    sleeve_pivot[c] = sleeve_pivot[c].map(pct_fmt)
display(sleeve_pivot.reset_index())


## 14) Concentration metrics


In [ ]:
concentration_rows = []
for s in STRATEGY_CFG["strategies"]:
    sub = weights_df[weights_df["strategy"] == s].sort_values("date").copy()
    weight_cols = [c for c in sub.columns if c.startswith("weight_")]
    for _, row in sub.iterrows():
        w = row[weight_cols].fillna(0.0).astype(float).values
        w = np.sort(w)[::-1]
        top1 = w[0] if len(w) else np.nan
        eff_n = 1 / np.sum(np.square(w)) if np.sum(np.square(w)) > 0 else np.nan
        concentration_rows.append({"date": row["date"], "strategy": s, "top1_weight": top1, "effective_n": eff_n})
concentration_df = pd.DataFrame(concentration_rows)
px.line(concentration_df, x="date", y="top1_weight", color="strategy", title="Largest single weight over time").show()
px.line(concentration_df, x="date", y="effective_n", color="strategy", title="Effective number of assets over time").show()


## 15) Turnover


In [ ]:
turnover_rows = []
for s in STRATEGY_CFG["strategies"]:
    sub = weights_df[weights_df["strategy"] == s].sort_values("date").copy()
    weight_cols = [c for c in sub.columns if c.startswith("weight_")]
    prev = None
    for _, row in sub.iterrows():
        cur = row[weight_cols].fillna(0.0).astype(float)
        turnover = np.nan if prev is None else np.abs(cur.values - prev.values).sum()
        turnover_rows.append({"date": row["date"], "strategy": s, "turnover": turnover})
        prev = cur
turnover_df = pd.DataFrame(turnover_rows)
px.line(turnover_df.dropna(), x="date", y="turnover", color="strategy", title="Turnover over time").show()
avg_turnover = turnover_df.groupby("strategy")["turnover"].mean().reset_index()
px.bar(avg_turnover, x="strategy", y="turnover", title="Average turnover").show()


## 16) Weight heatmap


In [ ]:
weight_cols = [c for c in weights_df.columns if c.startswith("weight_")]
cumw = weights_df[weight_cols].sum().sort_values(ascending=False).head(15).index.tolist()
heat = weights_df[["date", "strategy"] + cumw].copy()
heat["date"] = heat["date"].dt.strftime("%Y-%m-%d")
heat["row"] = heat["strategy"] + " | " + heat["date"]
heat = heat.set_index("row")[cumw]
heat.columns = [c.replace("weight_", "") for c in heat.columns]
px.imshow(heat.T, aspect="auto", title="Weight evolution heatmap (top 15 total weight columns)").show()


## 17) View diagnostics


In [ ]:
if len(diag_df):
    diag_show = diag_df.copy()
    if "final_view" in diag_show.columns:
        diag_show["view_pct"] = diag_show["final_view"].map(pct_fmt)
    if "confidence" in diag_show.columns:
        diag_show["confidence_pct"] = diag_show["confidence"].map(lambda x: f"{x*100:.0f}%")
    display(diag_show.tail(40))


## 18) What would I trust?


In [ ]:
best_final = summary_numeric.loc[summary_numeric["ending_value"].idxmax(), "strategy"]
best_dd = summary_numeric.loc[summary_numeric["maxdd"].idxmax(), "strategy"]
best_sharpe = summary_numeric.loc[summary_numeric["sharpe"].idxmax(), "strategy"]
print(f"Best pure growth: {best_final}")
print(f"Best drawdown control: {best_dd}")
print(f"Best risk-adjusted result: {best_sharpe}")
